## Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F

## Dataset Loading for NeRF

### Uses MipNeRF 360 bicycle data available in kaggle Repository

In [ ]:
class KaggleMipNeRFDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, poses_path, downsample_factor=4, device="cuda"):
        super().__init__()
        self.device = device
        self.downsample_factor = downsample_factor
        
        raw_poses_bounds = np.load(poses_path)
        poses = raw_poses_bounds[:, :-2].reshape([-1, 3, 5])
        bounds = raw_poses_bounds[:, -2:]
        
        H, W, focal = poses[0, :3, 4]
        self.H = int(H // downsample_factor)
        self.W = int(W // downsample_factor)
        self.focal = focal / downsample_factor
        
        self.poses = torch.from_numpy(poses[:, :3, :4]).float().to(device)
        self.bounds = torch.from_numpy(bounds).float().to(device)

        self.images = []
        image_files = sorted(os.listdir(images_dir))
        
        print(f"Pre-loading {len(image_files)} images into memory...")
        for img_name in image_files:
            img_path = os.path.join(images_dir, img_name)
            img = Image.open(img_path).convert("RGB")
            if downsample_factor > 1:
                img = img.resize((self.W, self.H), Image.Resampling.LANCZOS)
            
            img_tensor = torch.from_numpy(np.array(img)).float() / 255.0
            self.images.append(img_tensor.to(device))
            
        self.images = torch.stack(self.images) # Shape: [N, H, W, 3]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return {
            "image": self.images[idx],
            "pose": self.poses[idx],
            "bounds": self.bounds[idx]
        }

## Bounding the Dataset

In [ ]:
def apply_mipnerf360_contraction(x):
    norm = torch.norm(x, dim=-1, keepdim=True)
    contracted = torch.where(norm <= 1.0, x, (2.0 - 1.0 / norm) * (x / norm))
    return contracted

## Sine Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, L_pos=10, L_dir=4):
        super().__init__()
        self.freqs_pos = 2.0 ** torch.linspace(0, L_pos - 1, L_pos)
        self.freqs_dir = 2.0 ** torch.linspace(0, L_dir - 1, L_dir)

    def forward(self, x, freqs):
        out = [x]
        for freq in freqs.to(x.device):
            out.append(torch.sin(x * freq))
            out.append(torch.cos(x * freq))
        return torch.cat(out, dim=-1)

## Building the Model

In [ ]:
class NeRFMLP(nn.Module):
    def __init__(self, L_pos=10, L_dir=4, hidden_dim=256):
        super().__init__()
        self.encoder = PositionalEncoding(L_pos, L_dir)
        
        in_pos = 3 + (3 * 2 * L_pos)
        in_dir = 3 + (3 * 2 * L_dir)
        
        self.layer1 = nn.Linear(in_pos, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)
        self.layer5 = nn.Linear(hidden_dim + in_pos, hidden_dim) # Skip connection
        self.layer6 = nn.Linear(hidden_dim, hidden_dim)
        self.layer7 = nn.Linear(hidden_dim, hidden_dim)
        self.layer8 = nn.Linear(hidden_dim, hidden_dim)
        
        self.sigma_out = nn.Linear(hidden_dim, 1)
        self.feature_out = nn.Linear(hidden_dim, hidden_dim)
        
       
        self.rgb_layer1 = nn.Linear(hidden_dim + in_dir, hidden_dim // 2)
        self.rgb_out = nn.Linear(hidden_dim // 2, 3)

    def forward(self, points, dirs):
        contracted_points = apply_mipnerf360_contraction(points)
        
        pos_enc = self.encoder(contracted_points, self.encoder.freqs_pos)
        dir_enc = self.encoder(dirs, self.encoder.freqs_dir)
        
        h = F.relu(self.layer1(pos_enc))
        h = F.relu(self.layer2(h))
        h = F.relu(self.layer3(h))
        h = F.relu(self.layer4(h))
        h = F.relu(self.layer5(torch.cat([h, pos_enc], dim=-1))) 
        h = F.relu(self.layer6(h))
        h = F.relu(self.layer7(h))
        h = self.layer8(h)
        
        sigma = F.softplus(self.sigma_out(h)) # Density must be > 0
        geo_feature = self.feature_out(h)
        
        h_color = torch.cat([geo_feature, dir_enc], dim=-1)
        h_color = F.relu(self.rgb_layer1(h_color))
        rgb = torch.sigmoid(self.rgb_out(h_color)) # Colors bounded [0, 1]
        
        return rgb, sigma

## Ray Conversion Functions

In [ ]:
def get_rays(H, W, focal, pose):
    i, j = torch.meshgrid(torch.linspace(0, W-1, W), torch.linspace(0, H-1, H), indexing='xy')
    dirs = torch.stack([(i - W * 0.5) / focal, -(j - H * 0.5) / focal, -torch.ones_like(i)], -1).to(pose.device)
    

    rays_d = torch.sum(dirs[..., None, :] * pose[:3, :3], -1) 
    rays_o = pose[:3, 3].expand(rays_d.shape)
    
    return rays_o, rays_d

def render_rays(model, rays_o, rays_d, bounds, num_samples=64):
    """Volumetric rendering equation."""

    near, far = bounds[0], bounds[1]
    t_vals = torch.linspace(0., 1., steps=num_samples).to(rays_o.device)
    z_vals = near + (far - near) * t_vals
    

    if model.training:
        mids = .5 * (z_vals[..., 1:] + z_vals[..., :-1])
        upper = torch.cat([mids, z_vals[..., -1:]], -1)
        lower = torch.cat([z_vals[..., :1], mids], -1)
        t_rand = torch.rand(z_vals.shape).to(rays_o.device)
        z_vals = lower + (upper - lower) * t_rand


    pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals[..., :, None]
    

    dirs = rays_d[..., None, :].expand(pts.shape)
    

    pts_flat = pts.reshape(-1, 3)
    dirs_flat = dirs.reshape(-1, 3)
    
    rgb_flat, sigma_flat = model(pts_flat, dirs_flat)
    

    rgb = rgb_flat.reshape(pts.shape[:2] + (3,))
    sigma = sigma_flat.reshape(pts.shape[:2])
    

    dists = z_vals[..., 1:] - z_vals[..., :-1]
    dists = torch.cat([dists, torch.tensor([1e10]).expand(dists[..., :1].shape).to(rays_o.device)], -1)
    
    alpha = 1.0 - torch.exp(-sigma * dists)
    

    transmittance = torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1)).to(rays_o.device), 1. - alpha + 1e-10], -1), -1)[:, :-1]
    weights = alpha * transmittance
    
    rgb_map = torch.sum(weights[..., None] * rgb, -2)
    return rgb_map, sigma

## Loss Function

### This is a part of the composite loss function that decides the continuous field output from the MLP

In [ ]:
def calculate_tv_loss(sigma):
    diff_z = torch.abs(sigma[:, 1:] - sigma[:, :-1])
    return torch.mean(diff_z)

## Training the function

### GPU/ CPU check, Image directories and model loading to device, training on fixed epochs and evaluating the PSNR by minimizing the composite loss function

In [ ]:
def train_nerf():
    # Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    
    img_dir = "/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle/images"
    poses_path = "/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle/poses_bounds.npy"
    
    dataset = KaggleMipNeRFDataset(img_dir, poses_path, downsample_factor=4, device=device)
    model = NeRFMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    
    # Hyperparameters
    EPOCHS = 100
    BATCH_RAYS = 2048 
    LAMBDA_TV = 1e-4  
    
    best_psnr = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        total_psnr = 0
        

        for i in tqdm(range(len(dataset)), desc=f"Epoch {epoch+1}/{EPOCHS}"):
            data = dataset[i]
            target_img = data["image"]
            pose = data["pose"]
            bounds = data["bounds"]
    
            rays_o, rays_d = get_rays(dataset.H, dataset.W, dataset.focal, pose)
   
            rays_o_flat = rays_o.reshape(-1, 3)
            rays_d_flat = rays_d.reshape(-1, 3)
            target_img_flat = target_img.reshape(-1, 3)
            
            select_inds = torch.randperm(rays_o_flat.shape[0])[:BATCH_RAYS]
            rays_o_batch = rays_o_flat[select_inds]
            rays_d_batch = rays_d_flat[select_inds]
            target_rgb_batch = target_img_flat[select_inds]
            
            optimizer.zero_grad()
            
            # Forward Pass
            pred_rgb, pred_sigma = render_rays(model, rays_o_batch, rays_d_batch, bounds)
            
            # Composite Loss Formulation
            loss_color = F.mse_loss(pred_rgb, target_rgb_batch)
            loss_tv = calculate_tv_loss(pred_sigma)
            
            loss = loss_color + (LAMBDA_TV * loss_tv)

            loss.backward()
            optimizer.step()
      
            psnr = -10.0 * torch.log10(loss_color)
            total_loss += loss.item()
            total_psnr += psnr.item()
            
        avg_psnr = total_psnr / len(dataset)
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(dataset):.4f} | PSNR: {avg_psnr:.2f} dB")
        
        # Checkpoint Management
        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            torch.save(model.state_dict(), 'best_nerf_weights.pth')
            print(f"--> Saved new best model at Epoch {epoch+1} with PSNR {best_psnr:.2f}")

## Calling the training within main function

In [ ]:
if __name__ == "__main__":
    train_nerf()